In [1]:
#add libraries
%pip install pandas matplotlib numpy
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Note: you may need to restart the kernel to use updated packages.


### Read Data

In [2]:
def read_data():
    return [
        pd.read_csv('../data/development_data/awards_players.csv'),
        pd.read_csv('../data/development_data/coaches.csv'),
        pd.read_csv('../data/development_data/players.csv'),
        pd.read_csv('../data/development_data/players_teams.csv'),
        pd.read_csv('../data/development_data/series_post.csv'),
        pd.read_csv('../data/development_data/teams.csv'),
        pd.read_csv('../data/development_data/teams_post.csv')
    ]

awards_players, coaches, players, players_teams, series_post, teams, teams_post = read_data()

### Data Selection

In [ ]:
awards_players = awards_players.drop(columns=['lgID']) 
coaches = coaches.drop(columns=['lgID'])
players = players.drop(columns=[ 'college', 'collegeOther', 'deathDate', 'firstseason', 'lastseason'])
players_teams = players_teams.drop(columns=['lgID'])
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser']) #'round', 'series' too ??
teams = teams.drop(columns=['lgID', 'franchID', 'confID', 'divID', 'arena', 'name'])
teams_post = teams_post.drop(columns=['lgID'])

### Data Merging

In [4]:
#team metrics

data = pd.merge(teams, teams_post, on=['year', 'tmID'], how='left')

data['playoff_qualification'] = data['playoff'].apply(lambda x: 1.0 if x == 'Y' else 0.0)
data.drop(columns=['playoff'], inplace=True)
data.fillna({'W' : 0, 'L' : 0}, inplace=True)
    
#player stats
player_stats = players_teams.groupby(['tmID', 'year']).agg({
    'points': 'sum',
    'rebounds': 'sum',
    'assists': 'sum',
    'steals': 'sum',
    'blocks': 'sum',
    'turnovers': 'sum'
}).reset_index()

data = pd.merge(data, player_stats, on=['year', 'tmID'], how='left')

coach_stats = coaches.groupby(['year', 'tmID']).agg({
    'won': 'sum',
    'lost': 'sum',
    'post_wins': 'sum',
    'post_losses': 'sum'
}).reset_index()

data = pd.merge(data, coach_stats, on=["year", "tmID"], how="left")

#awards
# awards_count = awards_players.groupby(['playerID', 'year']).size().reset_index(name='num_awards')

# data = pd.merge(data, awards_count, on=["year", "playerID"], how="left")
# data['num_awards'] = data['num_awards'].fillna(0)

In [5]:
data['win_loss_ratio'] = data['won_x'] / (data['won_x'] + data['lost_x'])  
data['next_year_playoff_qualification'] = data.groupby('tmID')['playoff_qualification'].shift(-1)

columns_to_drop = ['tmID', 'lgID', 'rank', 'firstRound', 'semis', 'finals']
data = data.drop(columns=columns_to_drop, errors='ignore')

data.columns

Index(['year', 'seeded', 'o_fgm', 'o_fga', 'o_ftm', 'o_fta', 'o_3pm', 'o_3pa',
       'o_oreb', 'o_dreb', 'o_reb', 'o_asts', 'o_pf', 'o_stl', 'o_to', 'o_blk',
       'o_pts', 'd_fgm', 'd_fga', 'd_ftm', 'd_fta', 'd_3pm', 'd_3pa', 'd_oreb',
       'd_dreb', 'd_reb', 'd_asts', 'd_pf', 'd_stl', 'd_to', 'd_blk', 'd_pts',
       'tmORB', 'tmDRB', 'tmTRB', 'opptmORB', 'opptmDRB', 'opptmTRB', 'won_x',
       'lost_x', 'GP', 'homeW', 'homeL', 'awayW', 'awayL', 'confW', 'confL',
       'min', 'attend', 'W', 'L', 'playoff_qualification', 'points',
       'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'won_y',
       'lost_y', 'post_wins', 'post_losses', 'win_loss_ratio',
       'next_year_playoff_qualification'],
      dtype='object')

In [6]:
data.fillna(0, inplace=True)

### Model training

In [7]:
%pip install scikit-learn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVR

Note: you may need to restart the kernel to use updated packages.


#### Initialization 

In [101]:
#result lists
accuracy_scores = []
error_scores = []

#separate feature and target columns
feature_columns = [col for col in data.columns if col not in ['next_year_playoff_qualification']]
target_column = 'next_year_playoff_qualification'

#Create model
#model = DecisionTreeClassifier(random_state=21) #acc: 0.55 error: 0.59 || acc: 0.59  error: 0.55
#model = RandomForestClassifier(random_state=21) #acc: 0.60 error: 0.43 || acc: 0.64  error: 0.44
model = LogisticRegression(random_state=21)     #acc: 0.64 error: 0.39 || acc: 0.57  error: 0.44
#model = SVR()                                   #acc: 0.62 error: 0.44 || acc: 0.60  error: 0.45

#### Year training cycle

In [102]:
for year in sorted(data['year'].unique())[:-1]:  # Leave out the last year since it won't have a year after it for testing
    # Separate train and test data
    year_span = 9
    train_data = data[data['year'] <= year] if year < (year_span - 1) else data[(data['year'] <= year) & (data['year'] >= year - (year_span - 1))]
    test_data = data[data['year'] == year + 1]
    
    # If there's no data for the next year (e.g., last year in the dataset), skip
    if test_data.empty:
        continue

    # Split features and target
    X_train = train_data[feature_columns]
    y_train = train_data[target_column]

    X_test = test_data[feature_columns]
    y_test = test_data[target_column]
    
    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Train the model
    model.fit(X_train, y_train)

    # Make predictions on the test set
    y_pred_proba = model.predict_proba(X_test)[:,1]
    y_pred_proba = np.round(y_pred_proba * 8 / sum(y_pred_proba),2)  # Normalize to sum to 8

    y_pred = np.zeros_like(y_pred_proba)
    indices = np.argsort(y_pred_proba)[-8:]  # Get indices of top 8 probabilities
    y_pred[indices] = 1                      # and set them to 1

    # Calculate accuracy and error
    accuracy_scores.append(round(accuracy_score(y_test, y_pred),2))

    error_array = np.abs(y_pred_proba - y_test.values)
    error_score = round(sum(error_array) / len(error_array),2)
    error_scores.append(error_score)

    # Output results for each year
    print(f"Year {year} -> {year + 1}:")
    print(f"Results: \n predict: {y_pred_proba}\n label: \t {y_pred}\n expected: {y_test.values}\n error: \t {error_array}")
    print(f"  Accuracy: {accuracy_scores[-1]}")
    print(f"  Error: \t {round(sum(error_array), 2)} / {len(error_array)} ({error_score})")
    print("\n")

Year 1 -> 2:
Results: 
 predict: [0.82 0.81 0.47 0.75 0.17 0.83 0.78 0.15 0.74 0.07 0.22 0.05 0.82 0.31
 0.81 0.19]
 label: 	 [1. 1. 0. 1. 0. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0.]
 expected: [1. 0. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 1. 1.]
 error: 	 [0.18 0.81 0.47 0.25 0.83 0.17 0.78 0.15 0.26 0.07 0.22 0.05 0.82 0.69
 0.19 0.81]
  Accuracy: 0.62
  Error: 	 6.75 / 16 (0.42)


Year 2 -> 3:
Results: 
 predict: [0.98 0.66 0.67 0.34 0.66 0.55 0.36 0.12 0.71 0.02 0.57 0.05 0.28 0.33
 0.96 0.73]
 label: 	 [1. 1. 1. 0. 1. 0. 0. 0. 1. 0. 1. 0. 0. 0. 1. 1.]
 expected: [1. 1. 1. 1. 0. 1. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0.]
 error: 	 [0.02 0.34 0.33 0.66 0.66 0.45 0.36 0.88 0.71 0.02 0.57 0.05 0.72 0.33
 0.96 0.73]
  Accuracy: 0.44
  Error: 	 7.79 / 16 (0.49)


Year 3 -> 4:
Results: 
 predict: [0.3  0.91 0.8  0.09 0.92 0.44 0.91 0.31 0.62 0.01 0.18 0.89 0.91 0.69]
 label: 	 [0. 1. 1. 0. 1. 0. 1. 0. 1. 0. 0. 1. 1. 1.]
 expected: [0. 0. 1. 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1.]
 error: 	 [0.3  0.91 0.2  0.91 0

### End Results

In [103]:
print(f"Accuracy  {accuracy_scores}")
print(f"Error \t {[float(e) for e in error_scores]}")
print("\nAverage Performance Over All Years:")
print(f"  Average Accuracy: {sum(accuracy_scores[:-1]) / len(accuracy_scores[:-1]):.2f}")
print(f"  Average Error: \t {round(sum(error_scores[:-1]) / len(error_scores[:-1]),2)}")

Accuracy  [0.62, 0.44, 0.57, 0.69, 0.54, 0.43, 0.38, 0.57, 0.38]
Error 	 [0.42, 0.49, 0.5, 0.36, 0.53, 0.51, 0.53, 0.42, 0.62]

Average Performance Over All Years:
  Average Accuracy: 0.53
  Average Error: 	 0.47
